# 01 - The data layer

Milestone 1 of `vol-strategy-backtester` loads two option-chain vendors (OptionsDX for 2010-2023, DoltHub for 2019-2026), cleans them into one long table, `option_quotes`, and flags every quote problem instead of deleting the row. This notebook copies three small modules out of `src/volbt/data/` and runs them on hand-made rows, so you can see each rule work. It needs no market data.

This notebook is generated by `tools/build_notebooks.py`. Do not edit it by hand; edit `src/` and rebuild. The measured results of the full build are in `docs/milestones/01-data-layer.md`.

## Where the data goes

Vendor sources are only read, from `VOL_DATA_DIR` (default `~/data/vol-projects`). Everything M1 writes goes under the repo's git-ignored `data/` folder (override with `VOLBT_WORK_DIR`): downloads in `data/raw/`, tables in `data/store/`.

```text
uv run python tools/build_data.py --fetch --export-dolthub   # downloads, then build
uv run python tools/build_data.py                            # build from the downloads
```

## The OSI symbol

One 21-character string names one contract: root padded to 6, `yymmdd`, `C` or `P`, strike x 1000 in 8 digits. The cells below are copied from `src/volbt/data/osi.py`.

In [ ]:
import re
from datetime import date
from decimal import Decimal

import numpy as np
import pandas as pd

In [ ]:
ROOT_WIDTH = 6
STRIKE_SCALE = 1000
STRIKE_DIGITS = 8
MAX_STRIKE_MILLI = 10**STRIKE_DIGITS - 1
OSI_PATTERN = re.compile(r"^(?P<root>[A-Z0-9 ]{6})(?P<yymmdd>\d{6})(?P<side>[CP])(?P<strike>\d{8})$")

In [ ]:
def strike_to_milli(strike) -> int:
    """450.5 -> 450500. Raises ValueError if the strike is negative, too big, or not a whole number of thousandths."""
    # Decimal(str(x)) reads the shortest decimal that round-trips the float: 450.5 -> "450.5".
    milli = Decimal(str(strike)) * STRIKE_SCALE
    if milli != milli.to_integral_value():
        raise ValueError(f"strike {strike} is not a whole number of thousandths of a dollar")
    if not 0 <= milli <= MAX_STRIKE_MILLI:
        raise ValueError(f"strike {strike} does not fit in {STRIKE_DIGITS} digits of thousandths")
    return int(milli)

In [ ]:
def osi_symbol(root: str, expiry: date, call_put: str, strike) -> str:
    """("SPY", 2024-03-15, "C", 450.00) -> "SPY   240315C00450000"."""
    if not root or len(root) > ROOT_WIDTH or not re.fullmatch(r"[A-Z0-9]+", root):
        raise ValueError(f"root {root!r} must be 1 to {ROOT_WIDTH} upper-case letters or digits")
    if call_put not in ("C", "P"):
        raise ValueError(f"call_put must be 'C' or 'P', not {call_put!r}")
    if not 2000 <= expiry.year <= 2099:
        raise ValueError(f"expiry {expiry} is outside the two-digit years 2000-2099")
    return f"{root:<{ROOT_WIDTH}}{expiry:%y%m%d}{call_put}{strike_to_milli(strike):0{STRIKE_DIGITS}d}"

In [ ]:
def parse_osi(symbol: str) -> tuple[str, date, str, Decimal]:
    """The inverse: "SPY   240315C00450000" -> ("SPY", date(2024, 3, 15), "C", Decimal("450.000"))."""
    match = OSI_PATTERN.match(symbol)
    if match is None:
        raise ValueError(f"{symbol!r} is not a 21-character OSI symbol")
    yymmdd = match["yymmdd"]
    # date() raises ValueError for a month 13 or a 30 February.
    expiry = date(2000 + int(yymmdd[:2]), int(yymmdd[2:4]), int(yymmdd[4:]))
    # scaleb(-3) divides by 1000 exactly and keeps the three decimals.
    return match["root"].rstrip(), expiry, match["side"], Decimal(match["strike"]).scaleb(-3)

In [ ]:
def osi_symbols(expiry: pd.Series, call_put: pd.Series, strike: pd.Series, root: str = "SPY") -> pd.Series:
    """The same strings as osi_symbol, for whole columns at once (millions of rows)."""
    milli_float = np.asarray(strike, dtype=float) * STRIKE_SCALE
    milli = np.rint(milli_float).astype(np.int64)
    # A strike off the 1/1000 grid would round silently here; refuse it like the scalar version.
    if np.any(np.abs(milli_float - milli) > 1e-6) or np.any((milli < 0) | (milli > MAX_STRIKE_MILLI)):
        raise ValueError("a strike is negative, too big, or not a whole number of thousandths")
    if not set(pd.unique(call_put)) <= {"C", "P"}:
        raise ValueError("call_put must be 'C' or 'P'")
    days = pd.to_datetime(pd.Series(expiry).reset_index(drop=True))
    yymmdd = (days.dt.year % 100) * 10000 + days.dt.month * 100 + days.dt.day
    prefix = f"{root:<{ROOT_WIDTH}}"
    return (prefix + yymmdd.astype(str).str.zfill(6) + pd.Series(call_put).reset_index(drop=True).astype(str)
            + pd.Series(milli).astype(str).str.zfill(STRIKE_DIGITS))

In [ ]:
symbol = osi_symbol('SPY', date(2024, 3, 15), 'C', 450.00)
print(repr(symbol))
print(parse_osi(symbol))

## The 13-bit quality flags

Each problem is one bit, so one integer holds all of a quote's problems and each consumer tests the bits it cares about with `&`. Prices are compared in whole cents, so the boundaries are exact. Copied from `src/volbt/data/flags.py`.

In [ ]:
from datetime import date

import numpy as np
import pandas as pd

In [ ]:
# Bit 0 is the lowest bit. The name list is in bit order: FLAG_NAMES[b] has value 1 << b.
BID_NULL = 1 << 0             # bid is null
ASK_NULL = 1 << 1             # ask is null
ZERO_BID = 1 << 2             # bid = 0
CROSSED = 1 << 3              # bid > ask
LOCKED = 1 << 4               # bid = ask > 0
WIDE = 1 << 5                 # ask - bid > mid, i.e. relative spread > 100% (only when bid > 0)
STALE = 1 << 6                # bid and ask unchanged from the contract's previous snapshot while |dS/S| >= 0.5%
NON_SESSION_DATE = 1 << 7     # the vendor dated the row on a day that is not a SPY session (and it is dropped)
RELABELLED = 1 << 8           # RESERVED, never set: the relabel rule it marked was removed in M1 (round 2)
VOLUME_UNRELIABLE = 1 << 9    # volume null, or OptionsDX before 2021-01-04, or DoltHub
UNDERLYING_MISMATCH = 1 << 10  # |UNDERLYING_LAST / SPY close - 1| > 0.10%
VENDOR_IV_INVALID = 1 << 11   # the vendor's IV is null or <= 0 (renamed from VENDOR_IV_NULL in M1)
EXPIRY_DAY = 1 << 12          # tau = 0: the quote is on the expiry date

FLAG_NAMES = (
    "BID_NULL", "ASK_NULL", "ZERO_BID", "CROSSED", "LOCKED", "WIDE", "STALE", "NON_SESSION_DATE",
    "RELABELLED", "VOLUME_UNRELIABLE", "UNDERLYING_MISMATCH", "VENDOR_IV_INVALID", "EXPIRY_DAY",
)
FLAG_BITS = {name: bit for bit, name in enumerate(FLAG_NAMES)}

# Reserved for M2: quote_iv.iv_flags, a separate column in a separate table. Never set in M1.
IV_FAIL = 1 << 0
PARITY_OUTLIER = 1 << 1
FORWARD_FALLBACK = 1 << 2
IV_FLAG_NAMES = ("IV_FAIL", "PARITY_OUTLIER", "FORWARD_FALLBACK")

# The thresholds, written as exact fractions so the cent arithmetic below stays in integers.
STALE_MOVE_DENOMINATOR = 200        # |dS/S| >= 1/200 = 0.5%
MISMATCH_DENOMINATOR = 1000         # |u/S - 1| > 1/1000 = 0.10%
VOLUME_RELIABLE_FROM = date(2021, 1, 4)

In [ ]:
def flag_names(mask: int) -> list[str]:
    """The names of the bits set in `mask`, lowest bit first."""
    return [name for bit, name in enumerate(FLAG_NAMES) if mask & (1 << bit)]

In [ ]:
def cents(prices) -> np.ndarray:
    """Dollars to whole cents, as float64 so a null stays NaN (every comparison with NaN is False)."""
    return np.rint(np.asarray(pd.array(prices, dtype="Float64").to_numpy(dtype=float, na_value=np.nan)) * 100)

In [ ]:
def set_where(condition, bit: int) -> np.ndarray:
    return np.where(condition, bit, 0).astype(np.int32)

In [ ]:
def mid_flags(bid: np.ndarray, ask: np.ndarray) -> np.ndarray:
    """Bits 0-5, which need only the quote itself: bid and ask in cents (NaN for null)."""
    out = set_where(np.isnan(bid), BID_NULL)
    out |= set_where(np.isnan(ask), ASK_NULL)
    out |= set_where(bid == 0, ZERO_BID)
    out |= set_where(bid > ask, CROSSED)
    out |= set_where((bid == ask) & (bid > 0), LOCKED)
    # WIDE: ask - bid > mid = (ask + bid)/2, i.e. 2(ask - bid) > ask + bid. Only for a bid above
    # zero: a zero bid has its own flag, and 0.00/0.05 would otherwise be "200% wide" as well.
    out |= set_where((bid > 0) & (2 * (ask - bid) > ask + bid), WIDE)
    return out

In [ ]:
def quote_flags(frame: pd.DataFrame) -> np.ndarray:
    """The 13-bit mask for every row of a session-dated quote table.

    Needs the columns: bid, ask, prev_bid, prev_ask (the same contract's previous snapshot,
    null if none), spy_close, prev_spy_close, underlying_last, volume, vendor_iv, source,
    quote_date, expiry, and date_flags (the calendar's bits for the row's date; 0 on every
    session, and the calendar keeps no other date).
    """
    bid, ask = cents(frame["bid"]), cents(frame["ask"])
    out = np.asarray(frame["date_flags"], dtype=np.int32) | mid_flags(bid, ask)

    # STALE: the quote did not move although SPY moved at least 0.5% since the contract's last
    # snapshot. |S - S_prev| / S_prev >= 1/200  <=>  200 |S - S_prev| >= S_prev, in cents.
    spot, prev_spot = cents(frame["spy_close"]), cents(frame["prev_spy_close"])
    unchanged = (bid == cents(frame["prev_bid"])) & (ask == cents(frame["prev_ask"]))
    moved = STALE_MOVE_DENOMINATOR * np.abs(spot - prev_spot) >= prev_spot
    out |= set_where(unchanged & moved, STALE)

    # VOLUME_UNRELIABLE: null volume means "unknown", not zero (ADR-0002 §5).
    source = np.asarray(frame["source"])
    quote_day = pd.to_datetime(frame["quote_date"]).to_numpy()
    early = (source == "optionsdx") & (quote_day < np.datetime64(VOLUME_RELIABLE_FROM))
    volume_null = pd.isna(frame["volume"]).to_numpy()
    out |= set_where(volume_null | early | (source == "dolthub"), VOLUME_UNRELIABLE)

    # UNDERLYING_MISMATCH: |u - S| / S > 1/1000  <=>  1000 |u - S| > S. A null u (DoltHub) never trips it.
    underlying = cents(frame["underlying_last"])
    out |= set_where(MISMATCH_DENOMINATOR * np.abs(underlying - spot) > spot, UNDERLYING_MISMATCH)

    # VENDOR_IV_INVALID: null, or not above zero. OptionsDX has IVs from -0.0005 to -0.00001 and
    # exact zeros; DoltHub has zero vols on bid = ask = 0 quotes. A very high IV (above 5) is a
    # real, if extreme, number and is not flagged here; tools/report_m1.py counts it.
    iv = np.asarray(pd.array(frame["vendor_iv"], dtype="Float64").to_numpy(dtype=float, na_value=np.nan))
    out |= set_where(np.isnan(iv) | (iv <= 0), VENDOR_IV_INVALID)
    out |= set_where(pd.to_datetime(frame["expiry"]).to_numpy() == quote_day, EXPIRY_DAY)
    return out

In [ ]:
# ADR-0009 test 3: each quote must set exactly these bits.
quotes = [(0.00, 0.05), (1.10, 1.00), (2.00, 2.00), (0.10, 0.40), (1.00, 1.10)]
bits = mid_flags(cents([b for b, _ in quotes]), cents([a for _, a in quotes]))
for (bid, ask), mask in zip(quotes, bits):
    print(f'{bid:.2f} / {ask:.2f} -> {int(mask):3d} {flag_names(int(mask))}')

## The calendar rule

A session is a date with a SPY bar. A row dated on any other day is flagged and dropped, even when its underlying equals the previous session's close and that session has no snapshot of its own: OptionsDX's 2022-06-20 file does look like the 06-17 close, but it lists 06-20's contracts, so filing it under 06-17 would leak contracts first listed on 06-20. The date map keeps the previous session and its close as a diagnostic. Copied from `src/volbt/data/calendar.py` (its import of the flag constants is left out: the cells above define them).

In [ ]:
import bisect

import numpy as np
import pandas as pd

In [ ]:
def sessions_from_bars(bars: pd.DataFrame) -> pd.DataFrame:
    """date, close: one row per SPY session, ascending."""
    sessions = bars[["date", "close"]].sort_values("date").reset_index(drop=True)
    if sessions["date"].duplicated().any():
        raise ValueError("SPY bars have a duplicate date")
    return sessions

In [ ]:
def build_date_map(snapshots: pd.DataFrame, bars: pd.DataFrame) -> pd.DataFrame:
    """Decide every (source, vendor_date): keep it as a session, or drop it.

    `snapshots` has one row per (source, vendor_date) with that date's `underlying_last`
    (null if the vendor has none) and, optionally, its row count `rows`.
    Returns the same rows plus: action ("session" / "drop"), quote_date (the vendor date for a
    session, null when dropped), flags (NON_SESSION_DATE on a dropped date), and, for a dropped
    date, the diagnostic prev_session and prev_close (the session before it and its SPY close).
    """
    sessions = sessions_from_bars(bars)
    session_days = list(sessions["date"])
    close_on = dict(zip(sessions["date"], sessions["close"]))
    session_set = set(session_days)
    out = snapshots.copy().sort_values(["source", "vendor_date"]).reset_index(drop=True)
    if "rows" not in out.columns:
        out["rows"] = pd.NA

    actions, quote_dates, flag_values, prev_sessions, prev_closes = [], [], [], [], []
    for vendor_date in out["vendor_date"]:
        if vendor_date in session_set:
            actions.append("session"), quote_dates.append(vendor_date), flag_values.append(0)
            prev_sessions.append(None), prev_closes.append(np.nan)
            continue
        # bisect_left gives the index of the first session >= vendor_date; one before it is p.
        index = bisect.bisect_left(session_days, vendor_date) - 1
        prev = session_days[index] if index >= 0 else None
        actions.append("drop"), quote_dates.append(None), flag_values.append(NON_SESSION_DATE)
        prev_sessions.append(prev), prev_closes.append(close_on[prev] if prev is not None else np.nan)

    out["action"], out["quote_date"], out["flags"] = actions, quote_dates, flag_values
    out["prev_session"], out["prev_close"] = prev_sessions, prev_closes
    out["flags"] = out["flags"].astype("int32")
    return out

In [ ]:
def missing_sessions(snapshot_dates, bars: pd.DataFrame, start=None, end=None) -> list:
    """Sessions in [start, end] (default: the snapshot range) on which the source has no snapshot."""
    have = set(snapshot_dates)
    first = start or min(have)
    last = end or max(have)
    return [d for d in sessions_from_bars(bars)["date"] if first <= d <= last and d not in have]

In [ ]:
bars = pd.DataFrame({'date': [date(2022, 6, 16), date(2022, 6, 17), date(2022, 6, 21),
                              date(2022, 7, 1), date(2022, 7, 5)],
                     'close': [301.37, 302.11, 311.48, 318.52, 319.07]})
snapshots = pd.DataFrame({'source': ['optionsdx'] * 5,
                          'vendor_date': [date(2022, 6, 16), date(2022, 6, 20), date(2022, 6, 21),
                                          date(2022, 7, 1), date(2022, 7, 4)],
                          'underlying_last': [301.37, 302.11, 311.48, 318.52, 318.52]})
date_map = build_date_map(snapshots, bars)
print(date_map[['vendor_date', 'action', 'quote_date', 'flags', 'prev_session', 'prev_close']].to_string(index=False))